# Mini-Projet Séries Temporelles — Partie 5
## IA générative et LLM — AAPL & QQQ

**Prérequis** : avoir exécuté le notebook de la partie 4. Il écrit dans `../outputs/` les fichiers `results_part4.csv` et `state_part4.json` (résultats, prévisions J+1, dérive, paramètres GARCH) que ce notebook relit.

## Étape 0 — Configuration, reproductibilité et reprise des données de l'équipe

In [84]:
# === Configuration & imports ==========================================================
# Pourquoi en premier : tout ce qui conditionne la reproductibilité (graine, versions, chemins, hyper-paramètres)
# est centralisé ici pour qu'un correcteur puisse relancer l'étude à l'identique.
import os, re, json, time, warnings, platform
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # moins de messages TensorFlow
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"      # calculs plus reproductibles d'une machine à l'autre
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display, Markdown
from scipy import stats

import statsmodels, scipy, sklearn
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.api import VAR
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from scipy.stats import jarque_bera
from sklearn.model_selection import TimeSeriesSplit
from arch import arch_model
import arch as arch_pkg

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---- Paramètres généraux (modifiables) -----------------------------------------------
QUICK_RUN = False                      # True = test rapide (3 époques, 2 plis, 1 graine) : ne sert qu'à vérifier que tout tourne
SEED      = 42                         # graine unique → résultats reproductibles
CSV_PATH  = "/content/data_aapl_qqq.csv"   # même fichier que l'équipe (yf.download(...).to_csv)
OUT_DIR   = "../outputs"               # tableaux de résultats (réutilisables pour la partie 6)
LLM_DIR   = "../llm"                   # prompts et réponses des LLM (partie 5)
for d in (OUT_DIR, LLM_DIR): os.makedirs(d, exist_ok=True)

# ---- Hyper-paramètres des réseaux (justifiés dans la section 4.1) ---------------------
CFG = dict(
    windows=[10, 20],      # 10 j couvre les lags significatifs 7-9 vus dans l'ACF/VAR ; 20 j ≈ 1 mois de bourse (= fenêtre de la volatilité)
    units=[16, 32],        # petits réseaux : ~1 700 observations seulement → risque de surapprentissage
    dropout=0.2, lr=1e-3, batch=32,
    max_epochs=80, patience=8, val_frac=0.15,
    cv_splits=4, cv_test_size=200,     # 4 plis walk-forward de 200 jours (≈ 10 mois) chacun
    n_seeds=3)                          # moyenne de 3 initialisations → réduit la variance d'entraînement
if QUICK_RUN:
    CFG.update(windows=[10], units=[16], max_epochs=3, patience=2, cv_splits=2, cv_test_size=120, n_seeds=1)

tf.get_logger().setLevel("ERROR")
np.random.seed(SEED); keras.utils.set_random_seed(SEED)
try: tf.config.experimental.enable_op_determinism()
except Exception: pass
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (12, 4.5)

print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__,
      "| statsmodels", statsmodels.__version__, "| arch", arch_pkg.__version__,
      "| scikit-learn", sklearn.__version__, "| tensorflow", tf.__version__, "| keras", keras.__version__)
print("QUICK_RUN =", QUICK_RUN, "|", CFG)

Python 3.13.16 | numpy 2.1.3 | pandas 2.2.3 | statsmodels 0.15.0 | arch 8.0.0 | scikit-learn 1.6.1 | tensorflow 2.21.0 | keras 3.13.2
QUICK_RUN = False | {'windows': [10, 20], 'units': [16, 32], 'dropout': 0.2, 'lr': 0.001, 'batch': 32, 'max_epochs': 80, 'patience': 8, 'val_frac': 0.15, 'cv_splits': 4, 'cv_test_size': 200, 'n_seeds': 3}


In [85]:
# === Étape 0b — rechargement des données de l'équipe ==================================
# Pourquoi : rendre ce notebook exécutable seul, en reconstruisant EXACTEMENT les objets de l'équipe
# (mêmes formules : rendement log, MA20, volatilité 20 j, split chronologique 90/10).
def load_prices(csv_path=CSV_PATH):
    """Lit le CSV écrit par yf.download(...).to_csv (3 lignes d'en-tête : Price / Ticker / Date)."""
    if os.path.exists(csv_path):
        raw = pd.read_csv(csv_path, header=[0, 1], index_col=0)
        raw.index = pd.to_datetime(raw.index, errors="coerce")
        raw = raw[raw.index.notna()].astype(float)      # supprime la ligne parasite « Date »
        px, src = raw["Adj Close"], f"CSV local : {csv_path}"
    else:                                               # repli : même appel que l'équipe
        import yfinance as yf
        data = yf.download(["AAPL", "QQQ"], start="2019-01-01", end="2026-10-01", auto_adjust=False, progress=False)
        px, src = data["Adj Close"], "yfinance (téléchargement direct)"
    return px[["AAPL", "QQQ"]].sort_index(), src

prices, source = load_prices()
assert prices.index.is_monotonic_increasing and not prices.index.duplicated().any() and not prices.isna().any().any()
returns = np.log(prices / prices.shift(1)).dropna()                      # r_t = ln(P_t / P_{t-1})
dataset = pd.DataFrame({
    "AAPL": prices["AAPL"], "QQQ": prices["QQQ"],
    "AAPL_Return": returns["AAPL"], "QQQ_Return": returns["QQQ"],
    "AAPL_MA20": prices["AAPL"].rolling(20).mean(),
    "AAPL_Volatility20": returns["AAPL"].rolling(20).std()})
split_index = int(len(dataset) * 0.90)                                   # split CHRONOLOGIQUE (jamais aléatoire)
train, test = dataset.iloc[:split_index].copy(), dataset.iloc[split_index:].copy()

print("Source :", source)
print(f"Période : {dataset.index[0].date()} → {dataset.index[-1].date()} | n = {len(dataset)} | train = {len(train)} | test = {len(test)}")
print(f"Train jusqu'au {train.index[-1].date()} | test à partir du {test.index[0].date()}")
print("✔ identique au notebook de l'équipe (1947 / 1752 / 195)" if (len(dataset), len(train), len(test)) == (1947, 1752, 195)
      else "⚠ dimensions différentes de celles du notebook de l'équipe (1947 / 1752 / 195) : vérifier le fichier CSV")

Source : CSV local : /content/data_aapl_qqq.csv
Période : 2019-01-02 → 2026-09-30 | n = 1947 | train = 1752 | test = 195
Train jusqu'au 2025-12-18 | test à partir du 2025-12-19
✔ identique au notebook de l'équipe (1947 / 1752 / 195)


### Étape 0c — Rappel des résultats de l'équipe (recalculés ici)

**Pourquoi recalculer ?** (1) Ce notebook reste autonome. (2) Les résultats des sections 1-3.3 **justifient les choix de la partie 4**
(quelles entrées donner au réseau ? faut-il modéliser les rendements ou les prix ? faut-il une entrée de volatilité ?)
et **alimentent les prompts de la partie 5** avec des chiffres réels (et non retapés à la main → pas d'erreur de recopie).
Aucun nouveau choix de modèle n'est fait ici : ARIMA(0,1,0) et GARCH(1,1) sont ceux retenus par l'équipe.

In [86]:
def recap_classique(train, dataset):
    """Recalcule les résultats clés des sections 2 à 3.3 (échantillon d'APPRENTISSAGE uniquement)."""
    R = {}
    # -- Stationnarité (ADF H0 = racine unitaire ; KPSS H0 = stationnarité : hypothèses opposées → tests complémentaires)
    rows = []
    for col, nm in [("AAPL", "AAPL prix"), ("QQQ", "QQQ prix"), ("AAPL_Return", "AAPL rendement"), ("QQQ_Return", "QQQ rendement")]:
        s = train[col].dropna(); a = adfuller(s); k = kpss(s, regression="c", nlags="auto")
        rows.append(dict(serie=nm, adf_stat=a[0], adf_p=a[1], kpss_stat=k[0], kpss_p=k[1]))
    R["stationnarite"] = pd.DataFrame(rows).set_index("serie")
    # -- ARIMA(0,1,0) retenu par auto_arima (équipe) + diagnostic des résidus
    arima_res = ARIMA(train["AAPL"].values, order=(0, 1, 0)).fit()
    e = pd.Series(arima_res.resid[1:])
    lb = acorr_ljungbox(e, lags=[10, 20], return_df=True); jb = jarque_bera(e); ar = het_arch(e)
    R["arima"] = dict(aic=arima_res.aic, lb10_p=lb.loc[10, "lb_pvalue"], lb20_p=lb.loc[20, "lb_pvalue"],
                      jb_stat=jb[0], jb_p=jb[1], archlm_stat=ar[0], archlm_p=ar[1])
    # -- VAR sur les rendements + causalité de Granger
    var_model = VAR(train[["AAPL_Return", "QQQ_Return"]].dropna())
    sel = var_model.select_order(maxlags=20); p_aic = sel.selected_orders["aic"]; p_bic = sel.selected_orders["bic"]
    var_res = var_model.fit(p_aic)
    g_q2a = var_res.test_causality("AAPL_Return", ["QQQ_Return"], kind="f")   # H0 : QQQ ne cause pas AAPL
    g_a2q = var_res.test_causality("QQQ_Return", ["AAPL_Return"], kind="f")   # H0 : AAPL ne cause pas QQQ
    R["var"] = dict(lag_aic=int(p_aic), lag_bic=int(p_bic), stable=bool(var_res.is_stable()),
                    corr_resid=float(var_res.resid.corr().iloc[0, 1]),
                    granger_qqq_to_aapl=(g_q2a.test_statistic, g_q2a.pvalue),
                    granger_aapl_to_qqq=(g_a2q.test_statistic, g_a2q.pvalue))
    # -- Cointégration de Johansen sur les log-prix
    jo = coint_johansen(np.log(train[["AAPL", "QQQ"]].dropna()), det_order=0, k_ar_diff=1)
    R["johansen"] = dict(trace=jo.lr1.tolist(), crit5=jo.cvt[:, 1].tolist(), coint=bool(jo.lr1[0] > jo.cvt[0, 1]))
    # -- GARCH(1,1) sur les rendements AAPL (en %) + diagnostic des résidus standardisés
    r_pct = train["AAPL_Return"].dropna() * 100
    g = arch_model(r_pct, mean="Constant", vol="GARCH", p=1, q=1).fit(disp="off")
    z = g.std_resid.dropna(); lbg = acorr_ljungbox(z, lags=[10, 20], return_df=True); arg = het_arch(z); jbg = jarque_bera(z)
    P = g.params
    R["garch"] = dict(mu=P["mu"], omega=P["omega"], alpha=P["alpha[1]"], beta=P["beta[1]"],
                      persistence=P["alpha[1]"] + P["beta[1]"],
                      lb10_p=lbg.loc[10, "lb_pvalue"], lb20_p=lbg.loc[20, "lb_pvalue"],
                      archlm_p=arg[1], jb_stat=jbg[0], jb_p=jbg[1])
    return R, g

R, garch_res = recap_classique(train, dataset)
display(R["stationnarite"].round(4))
print("ARIMA(0,1,0)  :", {k: round(float(v), 4) for k, v in R["arima"].items()})
print("VAR           :", R["var"])
print("Johansen      :", R["johansen"])
print("GARCH(1,1)    :", {k: round(float(v), 4) for k, v in R["garch"].items()})

,adf_stat,adf_p,kpss_stat,kpss_p
serie,,,,
AAPL prix,-0.7299,0.8388,5.9976,0.01
QQQ prix,0.0065,0.9591,5.4912,0.01
AAPL rendement,-13.5753,0.0000,0.2135,0.10
QQQ rendement,-13.2064,0.0000,0.0903,0.10


ARIMA(0,1,0)  : {'aic': 8557.0746, 'lb10_p': 0.0656, 'lb20_p': 0.0704, 'jb_stat': 5140.8186, 'jb_p': 0.0, 'archlm_stat': 257.9666, 'archlm_p': 0.0}
VAR           : {'lag_aic': 9, 'lag_bic': 1, 'stable': True, 'corr_resid': 0.8154884979568195, 'granger_qqq_to_aapl': (np.float64(3.109370131211389), np.float64(0.0009914162824246797)), 'granger_aapl_to_qqq': (np.float64(1.0962004836727854), np.float64(0.3617962842416341))}
Johansen      : {'trace': [10.855268029330174, 0.485214292580505], 'crit5': [15.4943, 3.8415], 'coint': False}
GARCH(1,1)    : {'mu': 0.1753, 'omega': 0.1577, 'alpha': 0.1044, 'beta': 0.8505, 'persistence': 0.9549, 'lb10_p': 0.724, 'lb20_p': 0.7107, 'archlm_p': 0.5653, 'jb_stat': 466.2971, 'jb_p': 0.0}


### Reprise des résultats de la partie 4
Tableau des résultats (`results_df`), prévisions du jour suivant (`NEXT`), dérive d'apprentissage (`drift`) et GARCH(1,1) à paramètres figés (`garch_fixed`).

In [ ]:
# === Reprise des résultats de la partie 4 =============================================
import json
results_df = pd.read_csv(f"{OUT_DIR}/results_part4.csv", index_col=0)
with open(f"{OUT_DIR}/state_part4.json", encoding="utf-8") as f:
    _state = json.load(f)
NEXT  = _state["NEXT"]
drift = _state["drift"]
# GARCH(1,1) : mêmes paramètres figés sur le train que dans la partie 4, filtrage sur toute la série
garch_fixed = arch_model(returns["AAPL"] * 100, mean="Constant", vol="GARCH", p=1, q=1).fix(pd.Series(_state["garch_params"]))
display(results_df.round(4))
print("dérive :", round(drift, 6), "| prévisions J+1 :", {k: round(v, 5) for k, v in NEXT.items()})

# Partie 5 — IA générative et LLM

### Protocole suivi (et pourquoi)
L'énoncé demande d'utiliser un LLM **comme assistant d'analyse, avec un regard critique**. Le protocole est donc :

| Étape | Ce qu'on fait | Pourquoi |
|---|---|---|
| 0 | L'équipe rédige **sa propre interprétation experte AVANT de consulter le LLM** | pour pouvoir comparer honnêtement ensuite (sinon on est influencé) |
| 5.1 | Prompt A → **hypothèses de modèles** à partir des statistiques descriptives et des tests de stationnarité | demandé par l'énoncé |
| 5.2 | Prompt B → **explication vulgarisée** pour un décideur non statisticien | demandé |
| 5.3 | Prompt C → **recommandations simulées** ; le LLM doit **formuler lui-même les limites** avant de recommander, puis on les complète | demandé |
| 5.4 | **Comparaison humain / LLM** + journal des erreurs détectées | critère « esprit critique face au LLM » |

**Garde-fous contre l'hallucination statistique** : (1) les prompts sont **construits par le code** à partir des résultats réels (aucun chiffre retapé à la main) ; (2) on demande au LLM de ne rien inventer et de signaler les informations manquantes ; (3) une fonction **repère tout nombre de la réponse absent du prompt** (p-value ou statistique potentiellement inventée) ; (4) chaque nombre « dérivé » par le LLM est **recalculé** par le code.

**Traçabilité** : chaque prompt est sauvegardé dans `llm/prompt_X.txt` et chaque réponse intégrale dans `llm/response_X.md`.

## 5.1 — Génération d'hypothèses de modèles à partir des statistiques descriptives (Prompt A)

Le prompt A est **identique à ce qui a été envoyé** au LLM (Claude, Anthropic — conversation du 11/10/2026). Les chiffres sont ceux des sections 2 et 3 du notebook de l'équipe ; la cellule de contrôle ci-dessous vérifie qu'ils correspondent bien aux valeurs recalculées.

In [98]:
# === Utilitaires de la partie 5 : traçabilité des prompts / réponses et détection de chiffres suspects ===
def save_text(path, text):
    with open(path, "w", encoding="utf-8") as f: f.write(text)

def read_text(path):
    return open(path, encoding="utf-8").read() if os.path.exists(path) else None

def ask_llm(prompt, model="claude-sonnet-5-5", max_tokens=1800):
    """Appel automatique OPTIONNEL (si ANTHROPIC_API_KEY est définie). Sinon : copier-coller manuel (voir run_llm_step)."""
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key: return None
    try:
        import anthropic
        msg = anthropic.Anthropic(api_key=key).messages.create(model=model, max_tokens=max_tokens,
                                                              messages=[{"role": "user", "content": prompt}])
        return "".join(b.text for b in msg.content if b.type == "text")
    except Exception as e:
        print("Appel API impossible :", e); return None

_NUM = re.compile(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?")
def _numbers(text):
    t = text.replace("−", "-").replace("%", " %")
    t = re.sub(r"(\d),(\d{2,})(?![,\d])", r"\1.\2", t)               # virgule décimale française (pas les couples type GARCH(1,1))
    return [(m.group(), float(m.group())) for m in _NUM.finditer(t)]

def flag_unverified_numbers(response, prompt):
    """Nombres de la réponse ABSENTS du prompt (à ±0.5 % ou à l'arrondi près) : statistiques possiblement inventées ou dérivées.
    Comparaison en valeur absolue. Ignore les entiers triviaux (≤ 30, années) et les constantes usuelles (0.01, 0.05, 0.1, 1.96, 95, 100, 252)."""
    ref = [abs(v) for _, v in _numbers(prompt)]
    flagged = []
    for s, x in _numbers(response):
        if float(x).is_integer() and (abs(x) <= 30 or 1990 <= abs(x) <= 2100): continue
        x = abs(x)
        if x in {0.01, 0.05, 0.1, 0.10, 0.5, 1.96, 95, 100, 252}: continue
        dec = len(s.split(".")[1].lower().split("e")[0]) if "." in s else 0
        if not any(abs(x - y) <= 0.005 * y + 1e-12 or round(y, dec) == x for y in ref):
            if s not in flagged: flagged.append(s)
    return flagged

def run_llm_step(name, prompt, answer_text=None):
    """Sauvegarde le prompt, récupère la réponse (fichier existant > texte fourni > API > absente) et la contrôle."""
    save_text(f"{LLM_DIR}/prompt_{name}.txt", prompt)
    path = f"{LLM_DIR}/response_{name}.md"
    resp = answer_text if answer_text is not None else read_text(path)     # réponse intégrée au notebook > fichier collé par l'équipe
    if answer_text is not None: save_text(path, resp)
    if resp is None:
        resp = ask_llm(prompt)
        if resp: save_text(path, resp)
    display(Markdown(f"**Prompt {name}** (sauvegardé dans `{LLM_DIR}/prompt_{name}.txt`)")); print(prompt)
    if resp is None:
        display(Markdown(f"> ⚠ **Réponse du LLM non disponible.** Copier le prompt ci-dessus dans le LLM de votre choix, enregistrer sa réponse "
                         f"**intégrale** dans `{LLM_DIR}/response_{name}.md`, puis ré-exécuter cette cellule "
                         f"(ou définir la variable d'environnement `ANTHROPIC_API_KEY` pour un appel automatique)."))
        return None
    display(Markdown(f"---\n**Réponse du LLM — {name}** (intégrale, `{LLM_DIR}/response_{name}.md`)\n\n" + resp))
    bad = flag_unverified_numbers(resp, prompt)
    print("\n🔎 Nombres de la réponse absents du prompt (à vérifier un par un) :", bad if bad else "aucun")
    return resp

In [99]:
PROMPT_A = """Tu es un expert en séries temporelles financières. Je prépare un mini-projet académique (cours Séries Temporelles, Data Science & IA). Voici les statistiques de mes données ; je veux des HYPOTHÈSES DE MODÈLES À TESTER, pas des prévisions.

DONNÉES
- Séries : prix ajustés (Adj Close) journaliers de AAPL et QQQ, du 2019-01-02 au 2026-09-30 (1947 observations). Apprentissage : 1752 observations (jusqu'au 2025-12-18) ; test : 195 observations.
- Rendements logarithmiques (1946 observations, échantillon complet) :
  AAPL : moyenne 0.001123 ; écart-type 0.019309 ; min -0.137708 ; Q1 -0.007782 ; médiane 0.001301 ; Q3 0.011201 ; max 0.142617 ; skewness -0.1201 ; kurtosis en excès 6.5079
  QQQ  : moyenne 0.000828 ; écart-type 0.015073 ; min -0.127593 ; Q1 -0.006139 ; médiane 0.001404 ; Q3 0.008745 ; max 0.113356 ; skewness -0.3657 ; kurtosis en excès 7.2799
- Tests de stationnarité (échantillon d'apprentissage) :
  ADF (H0 = racine unitaire) : AAPL prix stat -0.7299 p 0.8388 ; QQQ prix stat 0.0065 p 0.9591 ; AAPL rendements stat -13.5753 p 2.2e-25 ; QQQ rendements stat -13.2064 p 1.1e-24
  KPSS (H0 = stationnarité autour d'une constante) : AAPL prix stat 5.9976 p 0.01 ; QQQ prix stat 5.4912 p 0.01 ; AAPL rendements stat 0.2135 p 0.10 ; QQQ rendements stat 0.0903 p 0.10 (p-values KPSS tronquées aux bornes de la table : 0.01 et 0.10)
- Lectures graphiques (approximatives) : ACF des prix : décroissance très lente (environ 0.88 au lag 40) ; ACF des rendements : autocorrélations faibles, lag 1 environ -0.08 (AAPL) et -0.13 (QQQ), quelques pics isolés aux lags 7 à 9 ; volatilité glissante 20 jours d'AAPL : regroupements marqués, pics vers 0.068 (mars 2020) et 0.05 (avril 2025), plancher vers 0.006.

DEMANDE
1. Propose une liste hiérarchisée d'hypothèses de modèles à tester (classiques d'abord, puis neuronaux). Pour chaque hypothèse : énoncé, justification chiffrée tirée UNIQUEMENT des données ci-dessus, test de validation à appliquer, risque principal.
2. Signale explicitement toute information manquante dont tu aurais besoin ; n'invente aucune statistique.
3. Termine par 3 limites de ta propre analyse."""

In [100]:
# Contrôle de cohérence : les chiffres du prompt A correspondent-ils aux valeurs recalculées ? (échantillon complet pour les descriptives, train pour les tests)
st = R["stationnarite"]
chk = []
for nm, col in [("AAPL", "AAPL"), ("QQQ", "QQQ")]:
    s = returns[col]
    chk += [(f"{nm} moyenne", s.mean()), (f"{nm} écart-type", s.std()), (f"{nm} skewness", s.skew()), (f"{nm} kurtosis excès", s.kurtosis())]
for nm in st.index:
    chk += [(f"ADF stat {nm}", st.loc[nm, "adf_stat"]), (f"KPSS stat {nm}", st.loc[nm, "kpss_stat"])]
live = pd.DataFrame(chk, columns=["valeur", "recalcul"]).set_index("valeur")
ref_nums = _numbers(PROMPT_A)          # (texte, valeur) : on compare à l'arrondi affiché dans le prompt
def _in_prompt(x):
    for s_, y in ref_nums:
        dec = len(s_.split(".")[1].lower().split("e")[0]) if "." in s_ else 0
        if round(x, dec) == y or abs(x - y) <= 1e-3 * abs(y): return True
    return False
live["présent dans le prompt A ?"] = [_in_prompt(x) for x in live["recalcul"]]
display(live.round(5))
print("✔ tous les chiffres du prompt A correspondent aux calculs de ce notebook" if live["présent dans le prompt A ?"].all()
      else "⚠ certaines valeurs diffèrent (données mises à jour par Yahoo ?) : régénérer le prompt A avec les nouvelles valeurs avant de le soumettre au LLM")

,recalcul,présent dans le prompt A ?
valeur,,
AAPL moyenne,0.00112,True
AAPL écart-type,0.01931,True
AAPL skewness,-0.12014,True
AAPL kurtosis excès,6.50793,True
QQQ moyenne,0.00083,True
QQQ écart-type,0.01507,True
QQQ skewness,-0.36574,True
QQQ kurtosis excès,7.27988,True
ADF stat AAPL prix,-0.72992,True


✔ tous les chiffres du prompt A correspondent aux calculs de ce notebook


In [101]:
RESPONSE_A = """**Remarque préalable.** Toutes les justifications ci-dessous reposent sur vos chiffres. Les valeurs que j'ai **calculées moi-même** à partir d'eux sont marquées *(dérivé)* : à recalculer de votre côté.

## A. Hypothèses classiques

**H1 — Les prix sont intégrés d'ordre 1 : modéliser les différences / rendements (d = 1).**
- Justification : ADF ne rejette pas la racine unitaire sur les prix (p = 0.8388 pour AAPL, p = 0.9591 pour QQQ) et KPSS rejette la stationnarité (statistiques 5.9976 et 5.4912, p = 0.01). C'est l'inverse sur les rendements (ADF p = 2.2e-25 et 1.1e-24 ; KPSS 0.2135 et 0.0903, p = 0.10). Les deux tests, d'hypothèses nulles opposées, convergent.
- Test : relancer ADF/KPSS avec constante puis constante + tendance ; contrôler le nombre de retards de l'ADF.
- Risque : ces tests ont une puissance limitée face à des ruptures (mars 2020) ; « d = 1 » est une approximation de travail, pas une preuve d'une racine unitaire exacte.

**H2 — Référence à battre : marche aléatoire, ARIMA(0,1,0).**
- Justification : l'ACF des prix décroît très lentement (≈ 0.88 au lag 40, signature d'une racine unitaire) et celle des rendements est faible (|ρ₁| ≤ 0.13).
- Test : Ljung-Box sur les résidus ; comparaison AIC/BIC avec ARIMA(1,1,0) et (0,1,1).
- Risque : si aucun modèle ne bat cette référence, ce n'est pas un échec mais le résultat attendu sur un actif liquide.
- Dérive : t ≈ 2.57 (AAPL) et ≈ 2.42 (QQQ) *(dérivé, calculé comme si les rendements étaient i.i.d.)* → inclure une constante dans les modèles de moyenne. Ce t est probablement surestimé car il ignore les queues épaisses et le regroupement de volatilité.

**H3 — Faible structure linéaire de court terme dans les rendements (AR d'ordre faible).**
- Justification : ρ₁ ≈ -0.08 (AAPL) et ≈ -0.13 (QQQ) sortent de la bande ±1.96/√1752 ≈ ±0.047 *(dérivé)*. Mais le R² associé est ≈ ρ² ≈ 0.64 % et ≈ 1.69 % *(dérivé)*.
- Test : ACF/PACF numériques, Ljung-Box, AIC/BIC d'AR(1), AR(2), ARMA(1,1).
- Risque : significatif ne veut pas dire exploitable (coûts de transaction) ; les pics aux lags 7-9 peuvent être des faux positifs (beaucoup de lags testés simultanément).

**H4 — Dépendance AAPL–QQQ : VAR sur les rendements + causalité de Granger.**
- Justification : le rapport des écarts-types est 1.28 *(dérivé)* et les deux séries ont des asymétries et kurtosis de même signe/ordre de grandeur. La corrélation n'est pas fournie : l'hypothèse est plausible mais non vérifiable ici.
- Test : sélection de l'ordre par AIC/BIC, stabilité, test de Granger dans les deux sens.
- Risque : une forte corrélation simultanée n'implique pas de prédictibilité décalée ; AIC et BIC choisissent souvent des ordres très différents.

**H5 — Cointégration des log-prix (Engle-Granger / Johansen).**
- Justification : condition nécessaire remplie si les deux séries sont I(1) (H1). Elle n'est pas suffisante, et vos données ne disent rien d'une relation de long terme. AAPL est une composante de QQQ (connaissance générale, **pas** issue de vos données).
- Test : Johansen (trace, constante dans la relation) ; si rejet de H0 seulement, estimer un VECM.
- Risque : la cointégration est souvent absente entre un titre et son indice sur 7 ans ; ne pas forcer un VECM.

## B. Volatilité et distribution

**H6 — Hétéroscédasticité conditionnelle : GARCH(1,1).**
- Justification : volatilité glissante entre ≈ 0.006 et ≈ 0.068, soit un facteur ≈ 11 *(dérivé)*, avec regroupements ; kurtosis en excès 6.5079 et 7.2799.
- Test : ARCH-LM sur les résidus du modèle de moyenne, puis sur les résidus standardisés ; Ljung-Box sur les carrés.
- Risque : un GARCH(1,1) gaussien sous-estime les queues.

**H7 — Innovations à queues épaisses (loi de Student).**
- Justification : pour une loi t i.i.d., kurtosis en excès = 6/(ν − 4), d'où ν ≈ 4.9 (AAPL) et ≈ 4.8 (QQQ) *(dérivé)*.
- Test : Jarque-Bera et QQ-plot des résidus standardisés ; estimer ν par maximum de vraisemblance ; comparer AIC/BIC gaussien vs Student.
- Risque : le kurtosis inconditionnel mélange regroupement de volatilité et vraies queues épaisses, donc ν conditionnel est en général plus grand ; il dépend aussi de 2-3 jours extrêmes (-0.138 et +0.143).

**H8 — Asymétrie (GJR-GARCH / EGARCH), surtout pour QQQ.**
- Justification : skewness -0.3657 (QQQ) contre -0.1201 (AAPL) ; médianes supérieures aux moyennes (0.001404 > 0.000828 ; 0.001301 > 0.001123).
- Test : test de biais de signe d'Engle-Ng ; significativité du paramètre d'asymétrie.
- Risque : une skewness inconditionnelle ne prouve pas un effet de levier ; pour AAPL l'indice est faible.

## C. Modèles neuronaux

**H9 — LSTM/GRU sur les rendements (stationnaires d'après H1), fenêtre ≥ 10 jours** (pics aux lags 7-9), entrées : rendements AAPL et QQQ + volatilité glissante 20 j.
- Test : validation walk-forward ; comparaison à la prévision naïve (rendement 0) ; test de Diebold-Mariano.
- Risque : si le R² linéaire plausible est ≤ 2 % (H3), le gain attendu est très faible ; avec ~1750 observations le surapprentissage est le danger principal.

**H10 — Hybride : sortie d'un GARCH comme variable d'entrée du réseau.**
- Test : étude d'ablation avec / sans la variable GARCH, mêmes hyper-paramètres.
- Risque : fuite d'information si la volatilité utilisée à la date t contient de l'information sur t+1 : bien décaler les séries.

**H11 — Pas de SARIMA a priori** : aucune donnée fournie n'indique de saisonnalité. À ne tester (jour de la semaine, lags 5 et 21) que si l'ACF numérique le suggère.

## Informations manquantes
Corrélation AAPL–QQQ ; valeurs numériques de l'ACF/PACF ; test de normalité sur les rendements ; test ARCH ; spécification déterministe et nombre de retards de l'ADF ; volumes ; composition de QQQ ; régime de marché de la période de test.

## Trois limites de mon analyse
1. Les statistiques descriptives semblent calculées sur l'échantillon complet (test inclus) : risque de regard vers l'avenir si elles guident le choix des modèles.
2. Les lectures graphiques (ACF, volatilité) sont approximatives.
3. Mes ν, R² et t-stats sont des approximations simples (dérivées) qui ne remplacent pas une estimation par maximum de vraisemblance ni des erreurs-types robustes."""
resp_A = run_llm_step("A", PROMPT_A, answer_text=RESPONSE_A)

**Prompt A** (sauvegardé dans `../llm/prompt_A.txt`)

Tu es un expert en séries temporelles financières. Je prépare un mini-projet académique (cours Séries Temporelles, Data Science & IA). Voici les statistiques de mes données ; je veux des HYPOTHÈSES DE MODÈLES À TESTER, pas des prévisions.

DONNÉES
- Séries : prix ajustés (Adj Close) journaliers de AAPL et QQQ, du 2019-01-02 au 2026-09-30 (1947 observations). Apprentissage : 1752 observations (jusqu'au 2025-12-18) ; test : 195 observations.
- Rendements logarithmiques (1946 observations, échantillon complet) :
  AAPL : moyenne 0.001123 ; écart-type 0.019309 ; min -0.137708 ; Q1 -0.007782 ; médiane 0.001301 ; Q3 0.011201 ; max 0.142617 ; skewness -0.1201 ; kurtosis en excès 6.5079
  QQQ  : moyenne 0.000828 ; écart-type 0.015073 ; min -0.127593 ; Q1 -0.006139 ; médiane 0.001404 ; Q3 0.008745 ; max 0.113356 ; skewness -0.3657 ; kurtosis en excès 7.2799
- Tests de stationnarité (échantillon d'apprentissage) :
  ADF (H0 = racine unitaire) : AAPL prix stat -0.7299 p 0.8388 ; QQQ prix stat 0.0

---
**Réponse du LLM — A** (intégrale, `../llm/response_A.md`)

**Remarque préalable.** Toutes les justifications ci-dessous reposent sur vos chiffres. Les valeurs que j'ai **calculées moi-même** à partir d'eux sont marquées *(dérivé)* : à recalculer de votre côté.

## A. Hypothèses classiques

**H1 — Les prix sont intégrés d'ordre 1 : modéliser les différences / rendements (d = 1).**
- Justification : ADF ne rejette pas la racine unitaire sur les prix (p = 0.8388 pour AAPL, p = 0.9591 pour QQQ) et KPSS rejette la stationnarité (statistiques 5.9976 et 5.4912, p = 0.01). C'est l'inverse sur les rendements (ADF p = 2.2e-25 et 1.1e-24 ; KPSS 0.2135 et 0.0903, p = 0.10). Les deux tests, d'hypothèses nulles opposées, convergent.
- Test : relancer ADF/KPSS avec constante puis constante + tendance ; contrôler le nombre de retards de l'ADF.
- Risque : ces tests ont une puissance limitée face à des ruptures (mars 2020) ; « d = 1 » est une approximation de travail, pas une preuve d'une racine unitaire exacte.

**H2 — Référence à battre : marche aléatoire, ARIMA(0,1,0).**
- Justification : l'ACF des prix décroît très lentement (≈ 0.88 au lag 40, signature d'une racine unitaire) et celle des rendements est faible (|ρ₁| ≤ 0.13).
- Test : Ljung-Box sur les résidus ; comparaison AIC/BIC avec ARIMA(1,1,0) et (0,1,1).
- Risque : si aucun modèle ne bat cette référence, ce n'est pas un échec mais le résultat attendu sur un actif liquide.
- Dérive : t ≈ 2.57 (AAPL) et ≈ 2.42 (QQQ) *(dérivé, calculé comme si les rendements étaient i.i.d.)* → inclure une constante dans les modèles de moyenne. Ce t est probablement surestimé car il ignore les queues épaisses et le regroupement de volatilité.

**H3 — Faible structure linéaire de court terme dans les rendements (AR d'ordre faible).**
- Justification : ρ₁ ≈ -0.08 (AAPL) et ≈ -0.13 (QQQ) sortent de la bande ±1.96/√1752 ≈ ±0.047 *(dérivé)*. Mais le R² associé est ≈ ρ² ≈ 0.64 % et ≈ 1.69 % *(dérivé)*.
- Test : ACF/PACF numériques, Ljung-Box, AIC/BIC d'AR(1), AR(2), ARMA(1,1).
- Risque : significatif ne veut pas dire exploitable (coûts de transaction) ; les pics aux lags 7-9 peuvent être des faux positifs (beaucoup de lags testés simultanément).

**H4 — Dépendance AAPL–QQQ : VAR sur les rendements + causalité de Granger.**
- Justification : le rapport des écarts-types est 1.28 *(dérivé)* et les deux séries ont des asymétries et kurtosis de même signe/ordre de grandeur. La corrélation n'est pas fournie : l'hypothèse est plausible mais non vérifiable ici.
- Test : sélection de l'ordre par AIC/BIC, stabilité, test de Granger dans les deux sens.
- Risque : une forte corrélation simultanée n'implique pas de prédictibilité décalée ; AIC et BIC choisissent souvent des ordres très différents.

**H5 — Cointégration des log-prix (Engle-Granger / Johansen).**
- Justification : condition nécessaire remplie si les deux séries sont I(1) (H1). Elle n'est pas suffisante, et vos données ne disent rien d'une relation de long terme. AAPL est une composante de QQQ (connaissance générale, **pas** issue de vos données).
- Test : Johansen (trace, constante dans la relation) ; si rejet de H0 seulement, estimer un VECM.
- Risque : la cointégration est souvent absente entre un titre et son indice sur 7 ans ; ne pas forcer un VECM.

## B. Volatilité et distribution

**H6 — Hétéroscédasticité conditionnelle : GARCH(1,1).**
- Justification : volatilité glissante entre ≈ 0.006 et ≈ 0.068, soit un facteur ≈ 11 *(dérivé)*, avec regroupements ; kurtosis en excès 6.5079 et 7.2799.
- Test : ARCH-LM sur les résidus du modèle de moyenne, puis sur les résidus standardisés ; Ljung-Box sur les carrés.
- Risque : un GARCH(1,1) gaussien sous-estime les queues.

**H7 — Innovations à queues épaisses (loi de Student).**
- Justification : pour une loi t i.i.d., kurtosis en excès = 6/(ν − 4), d'où ν ≈ 4.9 (AAPL) et ≈ 4.8 (QQQ) *(dérivé)*.
- Test : Jarque-Bera et QQ-plot des résidus standardisés ; estimer ν par maximum de vraisemblance ; comparer AIC/BIC gaussien vs Student.
- Risque : le kurtosis inconditionnel mélange regroupement de volatilité et vraies queues épaisses, donc ν conditionnel est en général plus grand ; il dépend aussi de 2-3 jours extrêmes (-0.138 et +0.143).

**H8 — Asymétrie (GJR-GARCH / EGARCH), surtout pour QQQ.**
- Justification : skewness -0.3657 (QQQ) contre -0.1201 (AAPL) ; médianes supérieures aux moyennes (0.001404 > 0.000828 ; 0.001301 > 0.001123).
- Test : test de biais de signe d'Engle-Ng ; significativité du paramètre d'asymétrie.
- Risque : une skewness inconditionnelle ne prouve pas un effet de levier ; pour AAPL l'indice est faible.

## C. Modèles neuronaux

**H9 — LSTM/GRU sur les rendements (stationnaires d'après H1), fenêtre ≥ 10 jours** (pics aux lags 7-9), entrées : rendements AAPL et QQQ + volatilité glissante 20 j.
- Test : validation walk-forward ; comparaison à la prévision naïve (rendement 0) ; test de Diebold-Mariano.
- Risque : si le R² linéaire plausible est ≤ 2 % (H3), le gain attendu est très faible ; avec ~1750 observations le surapprentissage est le danger principal.

**H10 — Hybride : sortie d'un GARCH comme variable d'entrée du réseau.**
- Test : étude d'ablation avec / sans la variable GARCH, mêmes hyper-paramètres.
- Risque : fuite d'information si la volatilité utilisée à la date t contient de l'information sur t+1 : bien décaler les séries.

**H11 — Pas de SARIMA a priori** : aucune donnée fournie n'indique de saisonnalité. À ne tester (jour de la semaine, lags 5 et 21) que si l'ACF numérique le suggère.

## Informations manquantes
Corrélation AAPL–QQQ ; valeurs numériques de l'ACF/PACF ; test de normalité sur les rendements ; test ARCH ; spécification déterministe et nombre de retards de l'ADF ; volumes ; composition de QQQ ; régime de marché de la période de test.

## Trois limites de mon analyse
1. Les statistiques descriptives semblent calculées sur l'échantillon complet (test inclus) : risque de regard vers l'avenir si elles guident le choix des modèles.
2. Les lectures graphiques (ACF, volatilité) sont approximatives.
3. Mes ν, R² et t-stats sont des approximations simples (dérivées) qui ne remplacent pas une estimation par maximum de vraisemblance ni des erreurs-types robustes.


🔎 Nombres de la réponse absents du prompt (à vérifier un par un) : ['2.57', '2.42', '0.047', '0.64', '1.69', '1.28', '4.9', '4.8']


### Contrôle des affirmations « dérivées » du LLM (recalcul)
Le LLM a produit quelques valeurs **qui ne figuraient pas dans le prompt** (marquées *dérivé* dans sa réponse). Une valeur plausible n'est pas une valeur exacte : on les **recalcule**.

In [102]:
rA, rQ = returns["AAPL"], returns["QQQ"]; n = len(rA)
der = pd.DataFrame([
    ("t de la dérive AAPL (i.i.d.)",            2.57, rA.mean() / (rA.std() / np.sqrt(n))),
    ("t de la dérive QQQ (i.i.d.)",             2.42, rQ.mean() / (rQ.std() / np.sqrt(n))),
    ("bande ±1.96/√1752 de l'ACF",              0.047, 1.96 / np.sqrt(1752)),
    ("R² d'un AR(1) avec ρ=-0.08 (en %)",       0.64, 100 * 0.08**2),
    ("R² d'un AR(1) avec ρ=-0.13 (en %)",       1.7,  100 * 0.13**2),
    ("rapport des écarts-types AAPL/QQQ",       1.28, rA.std() / rQ.std()),
    ("facteur pic/plancher de volatilité",      11,   0.068 / 0.006),
    ("ν Student implicite AAPL = 4 + 6/kurt",   4.9,  4 + 6 / rA.kurtosis()),
    ("ν Student implicite QQQ = 4 + 6/kurt",    4.8,  4 + 6 / rQ.kurtosis()),
], columns=["affirmation dérivée du LLM", "valeur LLM", "recalcul"]).set_index("affirmation dérivée du LLM")
der["écart relatif"] = (der["valeur LLM"] - der["recalcul"]).abs() / der["recalcul"].abs()
der["verdict"] = np.where(der["écart relatif"] < 0.05, "✔ exact (arrondi)", "✗ à corriger")
display(der.round(4))

,valeur LLM,recalcul,écart relatif,verdict
affirmation dérivée du LLM,,,,
t de la dérive AAPL (i.i.d.),2.570,2.5659,0.0016,✔ exact (arrondi)
t de la dérive QQQ (i.i.d.),2.420,2.4228,0.0012,✔ exact (arrondi)
bande ±1.96/√1752 de l'ACF,0.047,0.0468,0.0037,✔ exact (arrondi)
R² d'un AR(1) avec ρ=-0.08 (en %),0.640,0.6400,0.0000,✔ exact (arrondi)
R² d'un AR(1) avec ρ=-0.13 (en %),1.700,1.6900,0.0059,✔ exact (arrondi)
rapport des écarts-types AAPL/QQQ,1.280,1.2810,0.0008,✔ exact (arrondi)
facteur pic/plancher de volatilité,11.000,11.3333,0.0294,✔ exact (arrondi)
ν Student implicite AAPL = 4 + 6/kurt,4.900,4.9220,0.0045,✔ exact (arrondi)
ν Student implicite QQQ = 4 + 6/kurt,4.800,4.8242,0.0050,✔ exact (arrondi)


### Analyse critique de la réponse A (équipe)

**Là où le LLM converge avec nos résultats (vérifié sur nos sorties).**
- *H1/H2* — d = 1 et marche aléatoire : confirmés (ADF/KPSS ; `auto_arima` choisit (0,1,0)).
- *H4* — il a bien signalé que la corrélation manquait et que l'hypothèse n'était « pas vérifiable ici » ; notre Granger confirme ensuite un lien **QQQ → AAPL** (p = 0.001) mais pas l'inverse.
- *H5* — il prévenait que la cointégration est souvent absente : notre Johansen (trace 10.86 < 15.49) ne la détecte pas, donc pas de VECM. C'était toutefois une **intuition**, pas un résultat dérivé de ses données.
- *H6/H7* — GARCH puis loi de Student : notre GARCH gaussien supprime bien l'effet ARCH (p = 0.565) **mais Jarque-Bera rejette encore la normalité** des résidus standardisés (466.3) → l'hypothèse « queues épaisses » est **corroborée** et reste à implémenter (`arch_model(..., dist="t")`).
- Aucune p-value ni statistique **inventée** : tous les nombres sont soit dans le prompt, soit recalculés ci-dessus et exacts à l'arrondi.

**Là où il simplifie, exagère ou va plus loin que les données.**
1. *Dérive significative (t ≈ 2.5)* : calcul « comme si i.i.d. » — il le dit lui-même, mais cela ignore queues épaisses et regroupement de volatilité ; de plus, 2019-2026 est un marché haussier : ce n'est pas une propriété générale. Conclusion « mettre une constante » raisonnable, justification fragile.
2. *H3 (AR faible)* : il présente ρ₁ ≈ -0.08 comme une piste, alors que nos résidus ARIMA sont déjà non autocorrélés à 5 % et que le VAR ne s'accorde pas sur l'ordre (AIC : 9, BIC : 1). Hypothèse à faible valeur ajoutée.
3. *ν ≈ 5 par le kurtosis* : formule exacte pour une loi t i.i.d. mais **inadaptée ici** (le kurtosis inconditionnel mélange regroupement de volatilité et queues) ; il le signale, mais la valeur ne doit pas être reprise telle quelle : il faut estimer ν par maximum de vraisemblance.
4. *H8 (asymétrie GJR/EGARCH)* : déduite d'une skewness **inconditionnelle** (-0.37) ; ce n'est pas un test d'effet de levier. À valider par un test de biais de signe avant de conclure.
5. *Connaissance externe* : « AAPL est une composante de QQQ » ne vient pas des données ; vraie, mais à sourcer (non vérifiable par le notebook).
6. *Hiérarchie des hypothèses* : l'ordre proposé est subjectif (aucun critère chiffré).

**Limites de la méthode.** Les statistiques descriptives du prompt sont calculées sur l'**échantillon complet** (test inclus), comme dans le notebook de l'équipe : le LLM n'a pas pu déceler ce léger regard vers l'avenir, mais il l'a soupçonné (limite n°1). Le LLM utilisé est le même que celui qui a aidé à écrire ce notebook : ses réponses ne sont donc **pas indépendantes** du code → idéalement refaire le prompt A avec un second LLM.

## 5.2 — Explication vulgarisée pour un décideur non statisticien (Prompt B)

Le prompt est **construit automatiquement** à partir des résultats réels du notebook (étape 0c + partie 4). Le LLM doit expliquer sans jargon et **sans inventer de chiffre**.
Ensuite : (1) on enregistre la réponse intégrale dans `llm/response_B.md`, (2) on la relit en cherchant les erreurs (cellule de contrôle), (3) on l'inclut en annexe du rapport.

In [103]:
def _p(p): return f"{p:.2e}" if p < 1e-3 else f"{p:.3f}"

def build_prompt_B():
    a, v, g, j, st_ = R["arima"], R["var"], R["garch"], R["johansen"], R["stationnarite"]
    tab = results_df[["RMSE", "MAE", "MAPE", "RMSE_ret", "Hit", "p_DM_rend", "verdict_vs_ARIMA"]].round(5).to_string()
    return f"""Tu t'adresses à un directeur d'investissement qui n'est PAS statisticien. Explique-lui en français courant, en 350 mots maximum et avec une analogie parlante, les résultats d'un mini-projet de prévision sur l'action Apple (AAPL) et l'ETF QQQ (Nasdaq-100). N'invente AUCUN chiffre : utilise uniquement ceux fournis ci-dessous, et dis-le si une information manque.
Structure : (1) ce qu'on a fait ; (2) ce qu'on a découvert ; (3) ce que les modèles de prévision savent et ne savent pas faire ; (4) ce qu'il ne faut PAS conclure.

RÉSULTATS
- Données : prix journaliers du {dataset.index[0].date()} au {dataset.index[-1].date()} ; apprentissage {len(train)} jours, test {len(test)} jours.
- Les prix ne sont pas stationnaires (test ADF : p = {_p(st_.loc['AAPL prix','adf_p'])} pour AAPL) ; les rendements journaliers le sont (ADF p = {_p(st_.loc['AAPL rendement','adf_p'])}).
- Modèle de référence ARIMA(0,1,0) = « marche aléatoire » (demain = aujourd'hui). Ses erreurs ne sont pas autocorrélées (Ljung-Box p = {_p(a['lb10_p'])}) mais leur variance change dans le temps (test ARCH p = {_p(a['archlm_p'])}).
- Relation AAPL / QQQ (VAR, ordre {v['lag_aic']} par AIC) : QQQ aide à prévoir AAPL (Granger : F = {v['granger_qqq_to_aapl'][0]:.2f}, p = {_p(v['granger_qqq_to_aapl'][1])}) ; l'inverse non (F = {v['granger_aapl_to_qqq'][0]:.2f}, p = {_p(v['granger_aapl_to_qqq'][1])}). Corrélation des erreurs : {v['corr_resid']:.3f}.
- Cointégration (Johansen) : statistique de trace {j['trace'][0]:.2f} contre seuil 5 % de {j['crit5'][0]:.2f} → {'relation d équilibre détectée' if j['coint'] else 'pas de relation d équilibre de long terme détectée'}.
- Volatilité (GARCH(1,1)) : alpha = {g['alpha']:.3f}, beta = {g['beta']:.3f}, persistance alpha+beta = {g['persistence']:.3f} (les périodes agitées durent). Après modélisation, plus d'effet ARCH résiduel (p = {_p(g['archlm_p'])}), mais les queues restent plus épaisses qu'une loi normale (Jarque-Bera p = {_p(g['jb_p'])}).
- Réseaux de neurones (LSTM, GRU, avec ou sans variable GARCH) comparés à la marche aléatoire sur {len(test)} jours de test, prévision à 1 jour. RMSE_ret = erreur sur les rendements journaliers ; Hit = proportion de jours où le sens (hausse/baisse) est bien prévu ; p_DM_rend = p-value du test de Diebold-Mariano contre la marche aléatoire (< 0.05 = différence significative) :
{tab}
"""

PROMPT_B = build_prompt_B()
resp_B = run_llm_step("B", PROMPT_B)

**Prompt B** (sauvegardé dans `../llm/prompt_B.txt`)

Tu t'adresses à un directeur d'investissement qui n'est PAS statisticien. Explique-lui en français courant, en 350 mots maximum et avec une analogie parlante, les résultats d'un mini-projet de prévision sur l'action Apple (AAPL) et l'ETF QQQ (Nasdaq-100). N'invente AUCUN chiffre : utilise uniquement ceux fournis ci-dessous, et dis-le si une information manque.
Structure : (1) ce qu'on a fait ; (2) ce qu'on a découvert ; (3) ce que les modèles de prévision savent et ne savent pas faire ; (4) ce qu'il ne faut PAS conclure.

RÉSULTATS
- Données : prix journaliers du 2019-01-02 au 2026-09-30 ; apprentissage 1752 jours, test 195 jours.
- Les prix ne sont pas stationnaires (test ADF : p = 0.839 pour AAPL) ; les rendements journaliers le sont (ADF p = 2.18e-25).
- Modèle de référence ARIMA(0,1,0) = « marche aléatoire » (demain = aujourd'hui). Ses erreurs ne sont pas autocorrélées (Ljung-Box p = 0.066) mais leur variance change dans le temps (test ARCH p = 1.14e-49).
- Relation AAPL / QQQ (VAR

> ⚠ **Réponse du LLM non disponible.** Copier le prompt ci-dessus dans le LLM de votre choix, enregistrer sa réponse **intégrale** dans `../llm/response_B.md`, puis ré-exécuter cette cellule (ou définir la variable d'environnement `ANTHROPIC_API_KEY` pour un appel automatique).

## 5.3 — Recommandations simulées avec limites explicites (Prompt C)

**Cadre : exercice pédagogique de simulation, pas un conseil financier.** On demande au LLM de **formuler lui-même les limites et risques AVANT de recommander**, puis l'équipe les **complète de façon critique** (cellule de synthèse en 5.4).
Le prompt inclut les prévisions des modèles pour le jour suivant la fin des données, **leur fiabilité mesurée sur le test** (pour que le LLM ne les présente pas comme certaines) et les prévisions de volatilité du GARCH sur 10 jours.

In [104]:
def build_prompt_C():
    fc = garch_fixed.forecast(horizon=10).variance.iloc[-1].values          # variances prévues (%²) h = 1..10
    vol_daily = np.sqrt(fc); vol_cum10 = float(np.sqrt(fc.sum()))
    last_date, last_px = dataset.index[-1].date(), float(dataset["AAPL"].iloc[-1])
    lines = []
    for nm in results_df.index:
        lines.append(f"  - {nm} : rendement prévu J+1 = {100*NEXT[nm]:+.3f} % ; RMSE_ret test = {results_df.loc[nm,'RMSE_ret']:.5f} ; "
                     f"Hit test = {'n/a (prévoit 0)' if np.isnan(results_df.loc[nm,'Hit']) else format(results_df.loc[nm,'Hit'], '.3f')} ; verdict vs marche aléatoire = {results_df.loc[nm,'verdict_vs_ARIMA']}")
    v = R["var"]
    return f"""Cadre : exercice PÉDAGOGIQUE de simulation dans un cours de séries temporelles. Ce n'est PAS un conseil financier réel.
Voici des prévisions produites par plusieurs modèles sur l'action Apple (AAPL). Étapes OBLIGATOIRES, dans cet ordre :
1. AVANT toute recommandation, liste TOI-MÊME au moins 6 limites et risques de ces prévisions et de toute recommandation qui en découlerait.
2. Puis propose 3 recommandations d'investissement SIMULÉES (position, taille relative, horizon, condition d'arrêt), en citant pour chacune les chiffres ci-dessous sur lesquels elle s'appuie.
3. Ne présente jamais une prévision comme certaine. N'invente AUCUN chiffre ; si une information manque, dis-le.

DONNÉES
- Dernière observation : {last_date}, prix ajusté AAPL = {last_px:.2f}. Prévisions pour le jour de bourse suivant (J+1) :
{chr(10).join(lines)}
- Rendement quotidien moyen d'apprentissage : {100*drift:+.3f} % par jour.
- Volatilité quotidienne prévue par le GARCH(1,1), J+1 à J+10 (en %) : {', '.join(f'{x:.2f}' for x in vol_daily)} ; volatilité cumulée sur 10 jours = {vol_cum10:.2f} %.
- Lien avec l'indice : QQQ « Granger-cause » AAPL (F = {v['granger_qqq_to_aapl'][0]:.2f}, p = {_p(v['granger_qqq_to_aapl'][1])}), pas l'inverse.
- Le test a porté sur {len(test)} jours (du {test.index[0].date()} au {test.index[-1].date()}).
"""

PROMPT_C = build_prompt_C()
resp_C = run_llm_step("C", PROMPT_C)

**Prompt C** (sauvegardé dans `../llm/prompt_C.txt`)

Cadre : exercice PÉDAGOGIQUE de simulation dans un cours de séries temporelles. Ce n'est PAS un conseil financier réel.
Voici des prévisions produites par plusieurs modèles sur l'action Apple (AAPL). Étapes OBLIGATOIRES, dans cet ordre :
1. AVANT toute recommandation, liste TOI-MÊME au moins 6 limites et risques de ces prévisions et de toute recommandation qui en découlerait.
2. Puis propose 3 recommandations d'investissement SIMULÉES (position, taille relative, horizon, condition d'arrêt), en citant pour chacune les chiffres ci-dessous sur lesquels elle s'appuie.
3. Ne présente jamais une prévision comme certaine. N'invente AUCUN chiffre ; si une information manque, dis-le.

DONNÉES
- Dernière observation : 2026-09-30, prix ajusté AAPL = 333.02. Prévisions pour le jour de bourse suivant (J+1) :
  - ARIMA(0,1,0) / naïf : rendement prévu J+1 = +0.000 % ; RMSE_ret test = 0.01659 ; Hit test = n/a (prévoit 0) ; verdict vs marche aléatoire = référence
  - Marche aléatoire + dérive : rendeme

> ⚠ **Réponse du LLM non disponible.** Copier le prompt ci-dessus dans le LLM de votre choix, enregistrer sa réponse **intégrale** dans `../llm/response_C.md`, puis ré-exécuter cette cellule (ou définir la variable d'environnement `ANTHROPIC_API_KEY` pour un appel automatique).

## 5.4 — Comparaison interprétation experte ↔ LLM et journal des erreurs

### Synthèse de la comparaison (à compléter au fur et à mesure)

| Thème | Interprétation de l'équipe | LLM | Verdict |
|---|---|---|---|
| Stationnarité / d = 1 (A) | prix I(1), rendements I(0) | idem, + demande de contrôler la spécification déterministe | **converge** |
| Référence (A) | ARIMA(0,1,0) | idem, « à battre » | **converge** |
| Dépendance AAPL-QQQ (A) | Granger QQQ→AAPL, pas de cointégration | VAR/Granger plausible ; cointégration « souvent absente » | **converge** (le LLM pressent, nous démontrons) |
| Distribution des rendements (A) | JB rejette la normalité même après GARCH | propose Student | **converge** (reste à implémenter) |
| Dérive, AR(1), asymétrie (A) | non retenus comme pistes fortes | les met en avant | **le LLM sur-interprète** (voir analyse 5.1) |
| Vulgarisation (B) | — | `<à compléter après réponse B>` | `<converge / simplifie à l'excès / invente>` |
| Recommandations et limites (C) | — | `<à compléter après réponse C>` | `<…>` |

### Journal des erreurs / approximations détectées dans les productions du LLM

| # | Réponse | Affirmation du LLM | Vérification | Verdict |
|---|---|---|---|---|
| 1 | A | t de dérive ≈ 2.57 / 2.42 | recalculé (cellule ci-dessus) ; calcul i.i.d. | exact mais **non robuste** (ignore hétéroscédasticité) |
| 2 | A | ν ≈ 4.9 / 4.8 | recalculé ; formule de la loi t i.i.d. | exact mais **inadapté** (kurtosis inconditionnel) |
| 3 | A | effet de levier possible (QQQ) | skewness inconditionnelle ≠ test de biais de signe | **non démontré** |
| 4 | B | `<à compléter : relire chaque chiffre contre le prompt B ; signaler toute causalité non fondée>` | | |
| 5 | C | `<à compléter : une prévision est-elle présentée comme certaine ? un chiffre est-il inventé ?>` | | |

**Questions à se poser pour B et C (grille de relecture).** (a) Un nombre de la réponse est-il absent du prompt ? (la cellule le signale) ; (b) « corrélation » est-elle présentée comme « causalité » ? (c) un résultat **non significatif** (Diebold-Mariano) est-il présenté comme une victoire d'un modèle ? (d) le LLM a-t-il ignoré l'incertitude (intervalles, volatilité) ? (e) une recommandation dépend-elle d'une prévision de rendement dont la fiabilité mesurée est celle d'un tirage à pile ou face ?

### Pistes pour la discussion critique (section 8.4 du rapport)
- **Avantages** : rapidité d'idéation (liste d'hypothèses de modèles en quelques secondes) ; traduction en langage courant ; rappel de tests à ne pas oublier (spécification de l'ADF, test de biais de signe).
- **Inconvénients** : risque de **sur-interprétation** (transformer une skewness en « effet de levier ») ; chiffres plausibles mais non vérifiés ; **non indépendance** si le même LLM écrit le code et juge les résultats ; non reproductibilité exacte (réponses différentes à chaque appel) ; confiance excessive d'un lecteur non statisticien dans un texte fluide.
- **Bonne pratique retenue** : prompts construits par le code, demande explicite de limites, recalcul systématique des valeurs dérivées, détection automatique des nombres hors prompt, journal des erreurs.